We want to check how models perform with temporal split.

In [1]:
!pip install xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, classification_report,
    ConfusionMatrixDisplay, roc_curve, RocCurveDisplay
)
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Load df_artists and df_songs from CSV with all columns visible in head/sample output.

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

df_artists = pd.read_csv('/content/drive/MyDrive/Hitmaker project dataset/df_artists.csv')

print(df_artists.shape)

# Load df_songs from CSV.

df_songs = pd.read_csv('/content/drive/MyDrive/Hitmaker project dataset/df_songs.csv')

print(df_songs.shape)

(13655, 44)
(38383, 43)


In [4]:
# Create df_artists_final from df_artists:
# - drops non-feature/identifier columns
# - filters to artists whose first top-20 hit was between 2000 and 2019
# - keeps combined_major_genre_categories, renamed to combined_major_genre_categories_artist

cols_to_drop = [
    'musicbrainz_artist_id', 'musicbrainz_mbid', 'spotify_id',
    'performer_pre_normalized', 'last_charting_song_year', 'first_song_year',
    'years_active_on_charts', 'first_charting_song_position',
    'first_charting_song_duration', 'genre_tags_musicbrainz',
    'first_year_on_chart_songs', 'genre_tags_through_first_top_10_hit',
    'major_genre_categories_through_first_top_10_hit',
    '#_of_major_genre_categories_through_first_top_10_hit',
    'musicbrainz_major_genre_categories', 'musicbrainz_#_of_genres',
    'spotify_genres', 'spotify_major_genre_categories',
]

df_artists_final = (
    df_artists[df_artists['first_top_20_hit_year'].between(2000, 2019)]
    .drop(columns=[c for c in cols_to_drop if c in df_artists.columns])
    .rename(columns={'combined_major_genre_categories': 'combined_major_genre_categories_artist'})
    .reset_index(drop=True)
)

print(df_artists_final.shape)
df_artists_final.head()

(778, 26)


,name,first_top_20_hit_year,first_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,top_20_hit_song_#_wks_on_chart_any_position,combined_major_genre_categories_artist,first_top_20_song_major_genres,first_top_20_song_duration_ms,first_top_20_song_acousticness,first_top_20_song_danceability,first_top_20_song_energy,first_top_20_song_instrumentalness,first_top_20_song_liveness,first_top_20_song_loudness,first_top_20_song_speechiness,first_top_20_song_tempo,first_top_20_song_valence,first_top_20_song_mode,first_top_20_song_explicit,degree_centrality_top20_rolling5,harmonic_closeness_centrality_top20_rolling5,betweenness_centrality_top20_rolling5,eigenvector_centrality_top20_rolling5,power_of_connected_artists_top20_rolling5,top_20_hitmaker
0,2 chainz,2012.0,2012.0,1.0,9.0,27.0,Hip Hop/Rap,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006764,5314.363492,0.010130,1.510211e-01,7104.5,1.0
1,21 savage,2017.0,2016.0,2.0,15.0,41.0,Hip Hop/Rap,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002871,5233.436039,0.001326,7.977090e-02,3116.5,1.0
2,3 doors down,2000.0,2000.0,1.0,2.0,53.0,"Metal, Pop, Rock","Blues, Metal, Rock",233933.0,0.00664,0.545,0.865,0.000011,0.168,-5.708,0.0286,99.009,0.543,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0
3,3oh!3,2008.0,2008.0,1.0,1.0,37.0,"Electronic/Dance, Hip Hop/Rap, Pop, Punk/Hardc...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,5 seconds of summer,2014.0,2014.0,1.0,6.0,20.0,"Pop, Punk/Hardcore",Pop,237248.0,0.02830,0.572,0.499,0.000000,0.213,-5.237,0.0311,101.593,0.108,1.0,0.0,0.000048,1.000000,0.000000,1.555431e-18,NaN,1.0


In [5]:
# One-hot encode combined_major_genre_categories_artist into binary columns in df_artists_final.
# Splits comma-separated genre strings, creates one binary column per genre,
# prefixed with 'artist_genre_', replacing the original column.
# Also adds #_of_genres_artist (count of distinct genres per artist) after artist_genre_unknown.

# Compute genre count before dropping the original column
genre_counts = (
    df_artists_final['combined_major_genre_categories_artist']
    .fillna('')
    .apply(lambda x: len([g for g in x.split(', ') if g.strip()]) if x else 0)
)

artist_genre_dummies = (
    df_artists_final['combined_major_genre_categories_artist']
    .fillna('')
    .str.split(', ')
    .explode()
    .str.strip()
    .pipe(lambda s: pd.get_dummies(s, prefix='artist_genre'))
    .groupby(level=0)
    .max()
)

# Remove the empty-string column if it exists (from NaN rows)
artist_genre_dummies = artist_genre_dummies.loc[:, artist_genre_dummies.columns != 'artist_genre_']

# Insert genre columns where combined_major_genre_categories_artist was, then drop original
insert_at = df_artists_final.columns.get_loc('combined_major_genre_categories_artist')
df_artists_final = pd.concat([
    df_artists_final.iloc[:, :insert_at],
    artist_genre_dummies,
    df_artists_final.iloc[:, insert_at + 1:]
], axis=1)

# Add binary flag for artists with no genre data — inserted after last genre column
all_genre_cols = [c for c in df_artists_final.columns if c.startswith(('artist_genre_', 'song_genre_'))]
last_genre_idx = df_artists_final.columns.get_loc(all_genre_cols[-1])
df_artists_final.insert(last_genre_idx + 1, 'artist_genre_unknown', (df_artists_final[all_genre_cols].sum(axis=1) == 0).astype(int))

# Insert #_of_genres_artist after artist_genre_unknown
unknown_idx = df_artists_final.columns.get_loc('artist_genre_unknown')
df_artists_final.insert(unknown_idx + 1, '#_of_genres_artist', genre_counts)

print("Artist genre columns added:", artist_genre_dummies.columns.tolist())
print("df_artists_final shape:", df_artists_final.shape)

Artist genre columns added: ['artist_genre_Blues', 'artist_genre_Classical', 'artist_genre_Country/Americana', 'artist_genre_Easy Listening/Vocal', 'artist_genre_Electronic/Dance', 'artist_genre_Experimental/Avant-Garde', 'artist_genre_Folk', 'artist_genre_Gospel/Christian/Religious', 'artist_genre_Hip Hop/Rap', 'artist_genre_Jazz', 'artist_genre_Latin', 'artist_genre_Metal', 'artist_genre_Pop', 'artist_genre_Punk/Hardcore', 'artist_genre_R&B/Soul/Funk', 'artist_genre_Reggae/Caribbean', 'artist_genre_Rock', 'artist_genre_World Music']
df_artists_final shape: (778, 45)


In [6]:
# Drop rows with null top_20_hitmaker (artists with no verified top-20 songs)
# and any duplicate rows, then confirm shape and class balance.

df_artists_final = (
    df_artists_final
    .dropna(subset=['top_20_hitmaker'])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(df_artists_final.shape)
print()
print(df_artists_final['top_20_hitmaker'].value_counts())

(759, 45)

top_20_hitmaker
0.0    431
1.0    328
Name: count, dtype: int64


In [7]:
# Drop name column, cast bool genre columns to int (0/1) in df_artists_final.

genre_cols = [c for c in df_artists_final.columns if c.startswith('genre_')]

df_artists_final = df_artists_final.drop(columns=['name'])
df_artists_final[genre_cols] = df_artists_final[genre_cols].astype(int)

print(df_artists_final.shape)
print(df_artists_final[genre_cols].dtypes.unique())

(759, 44)
[]


In [8]:
# Impute missing values in df_artists_final and drop Spotify audio features:
# - Network metrics → fill with 0 (no collaboration data = no centrality)
# - Weeks on chart → fill with median
# - Spotify audio features → dropped entirely

network_metric_cols = [
    'degree_centrality', 'harmonic_closeness', 'betweenness_centrality',
    'eigenvector_centrality', 'power'
]
for col in network_metric_cols:
    if col in df_artists_final.columns:
        df_artists_final[col] = df_artists_final[col].fillna(0)

wks_col = 'top_20_hit_song_#_wks_on_chart_any_position'
if wks_col in df_artists_final.columns:
    median_wks = df_artists_final[wks_col].median()
    df_artists_final[wks_col] = df_artists_final[wks_col].fillna(median_wks)
    print(f"Weeks on chart median impute value: {median_wks}")

spotify_audio_cols = [
    'first_top_20_song_duration_ms', 'first_top_20_song_acousticness',
    'first_top_20_song_danceability', 'first_top_20_song_energy',
    'first_top_20_song_instrumentalness', 'first_top_20_song_liveness',
    'first_top_20_song_loudness', 'first_top_20_song_speechiness',
    'first_top_20_song_tempo', 'first_top_20_song_valence',
    'first_top_20_song_mode', 'first_top_20_song_key',
    'first_top_20_song_explicit', 'first_top_20_song_popularity',
]
df_artists_final = df_artists_final.drop(
    columns=[c for c in spotify_audio_cols if c in df_artists_final.columns]
)

print("Imputation complete. Spotify audio features dropped.")
print(f"df_artists_final shape: {df_artists_final.shape}")
print(df_artists_final.isnull().sum()[df_artists_final.isnull().sum() > 0])

Weeks on chart median impute value: 26.0
Imputation complete. Spotify audio features dropped.
df_artists_final shape: (759, 32)
first_top_20_song_major_genres                  391
degree_centrality_top20_rolling5                206
harmonic_closeness_centrality_top20_rolling5    206
betweenness_centrality_top20_rolling5           206
eigenvector_centrality_top20_rolling5           206
power_of_connected_artists_top20_rolling5       246
dtype: int64


In [9]:
# Fill null network metric columns with 0 (no collaboration data = no centrality)
# degree_centrality and power dropped due to collinearity — not used in model
df_artists_final = df_artists_final.drop(columns=[
    'degree_centrality_top20_rolling5',
    'power_of_connected_artists_top20_rolling5',
], errors='ignore')

network_cols = [
    'harmonic_closeness_centrality_top20_rolling5',
    'betweenness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
]
df_artists_final[network_cols] = df_artists_final[network_cols].fillna(0)
print(df_artists_final[network_cols].isnull().sum())

harmonic_closeness_centrality_top20_rolling5    0
betweenness_centrality_top20_rolling5           0
eigenvector_centrality_top20_rolling5           0
dtype: int64


In [10]:
df_artists_final.head()

,first_top_20_hit_year,first_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,top_20_hit_song_#_wks_on_chart_any_position,artist_genre_Blues,artist_genre_Classical,artist_genre_Country/Americana,artist_genre_Easy Listening/Vocal,artist_genre_Electronic/Dance,artist_genre_Experimental/Avant-Garde,artist_genre_Folk,artist_genre_Gospel/Christian/Religious,artist_genre_Hip Hop/Rap,artist_genre_Jazz,artist_genre_Latin,artist_genre_Metal,artist_genre_Pop,artist_genre_Punk/Hardcore,artist_genre_R&B/Soul/Funk,artist_genre_Reggae/Caribbean,artist_genre_Rock,artist_genre_World Music,artist_genre_unknown,#_of_genres_artist,first_top_20_song_major_genres,harmonic_closeness_centrality_top20_rolling5,betweenness_centrality_top20_rolling5,eigenvector_centrality_top20_rolling5,top_20_hitmaker
0,2012.0,2012.0,1.0,9.0,27.0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,0,1,NaN,5314.363492,0.010130,1.510211e-01,1.0
1,2017.0,2016.0,2.0,15.0,41.0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,0,1,NaN,5233.436039,0.001326,7.977090e-02,1.0
2,2000.0,2000.0,1.0,2.0,53.0,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,True,False,0,3,"Blues, Metal, Rock",0.000000,0.000000,0.000000e+00,1.0
3,2008.0,2008.0,1.0,1.0,37.0,False,False,False,False,True,False,False,False,True,False,False,False,True,True,False,False,True,False,0,5,NaN,0.000000,0.000000,0.000000e+00,1.0
4,2014.0,2014.0,1.0,6.0,20.0,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,0,2,Pop,1.000000,0.000000,1.555431e-18,1.0


In [11]:
# Drop columns not used as model features
# Save year BEFORE dropping — needed for temporal split in Cell 17
year_for_split = df_artists_final['first_top_20_hit_year'].copy()

cols_to_drop = ['first_top_20_song_major_genres', 'first_top_20_hit_year', 'first_charting_song_year']
df_artists_final = df_artists_final.drop(columns=[c for c in cols_to_drop if c in df_artists_final.columns])
print(df_artists_final.shape)

(759, 27)


In [12]:
# Separate features and target
X = df_artists_final.drop(columns=['top_20_hitmaker'])
y = df_artists_final['top_20_hitmaker']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny value counts:\n{y.value_counts()}")
print(f"\ny class balance:\n{y.value_counts(normalize=True).round(3)}")

X shape: (759, 26)
y shape: (759,)

y value counts:
top_20_hitmaker
0.0    431
1.0    328
Name: count, dtype: int64

y class balance:
top_20_hitmaker
0.0    0.568
1.0    0.432
Name: proportion, dtype: float64


In [13]:
year_for_split.value_counts().sort_index()

,count
first_top_20_hit_year,
2000.0,35
2001.0,44
2002.0,40
2003.0,45
2004.0,41
2005.0,40
2006.0,51
2007.0,42
2008.0,27


In [14]:
# Temporal train/test split: train on 2000-2014, test on 2015-2019
train_mask = year_for_split <= 2014
test_mask  = year_for_split >= 2015

X_train = X[train_mask.values]
X_test  = X[test_mask.values]
y_train = y[train_mask.values]
y_test  = y[test_mask.values]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nTrain years: 2000–2014")
print(f"Test years:  2015–2019")
print(f"\nTrain class balance:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest class balance:\n{y_test.value_counts(normalize=True).round(3)}")

# Check test set size
print(f"\nTest set size: {len(y_test)}")
print(y_test.value_counts())

Train: (590, 26), Test: (169, 26)

Train years: 2000–2014
Test years:  2015–2019

Train class balance:
top_20_hitmaker
0.0    0.58
1.0    0.42
Name: proportion, dtype: float64

Test class balance:
top_20_hitmaker
0.0    0.527
1.0    0.473
Name: proportion, dtype: float64

Test set size: 169
top_20_hitmaker
0.0    89
1.0    80
Name: count, dtype: int64


In [15]:
# Scale features for models sensitive to feature magnitude (Logistic Regression, SVM)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")

Scaling complete.
X_train_scaled shape: (590, 26)


In [16]:
# Evaluate models using 5-fold cross-validation on training set only.
# Logistic Regression and SVM wrapped in pipelines with StandardScaler to ensure convergence.

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import numpy as np

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

models = {
    'Dummy':               DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=42))]),
    'SVM':                 Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
}

results = []
for name, model in models.items():
    cv = cross_validate(model, X_train, y_train, cv=5, scoring=scoring, return_train_score=True)
    results.append({
        'Model':           name,
        'Accuracy':        cv['test_accuracy'].mean(),
        'Precision':       cv['test_precision'].mean(),
        'Recall':          cv['test_recall'].mean(),
        'F1':              cv['test_f1'].mean(),
        'ROC-AUC (CV)':    cv['test_roc_auc'].mean(),
        'ROC-AUC (Train)': cv['train_roc_auc'].mean(),
        'Overfit Gap':     cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean(),
    })

df_results = pd.DataFrame(results).set_index('Model').round(3)
print(df_results)

                     Accuracy  Precision  Recall     F1  ROC-AUC (CV)  \
Model                                                                   
Dummy                   0.532      0.442   0.427  0.434         0.518   
Logistic Regression     0.656      0.621   0.488  0.543         0.701   
SVM                     0.668      0.657   0.443  0.528         0.714   
Random Forest           0.673      0.621   0.585  0.600         0.723   
XGBoost                 0.654      0.594   0.589  0.590         0.701   

                     ROC-AUC (Train)  Overfit Gap  
Model                                              
Dummy                          0.506       -0.012  
Logistic Regression            0.768        0.068  
SVM                            0.856        0.142  
Random Forest                  0.988        0.265  
XGBoost                        0.995        0.294  


In [17]:
# Lightweight regularization pass for Random Forest and XGBoost.
# Goal: close overfit gaps before committing to full hyperparameter search.
# 20 iterations each, focused on depth and leaf/child weight controls.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

# --- Random Forest ---
rf_reg_grid = {
    'max_depth':        randint(3, 8),
    'min_samples_leaf': randint(10, 50),
    'min_samples_split':randint(10, 50),
    'max_features':     ['sqrt', 'log2', 0.5],
}

rf_reg_search = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    rf_reg_grid,
    n_iter=20, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
rf_reg_search.fit(X_train, y_train)

# --- XGBoost ---
xgb_reg_grid = {
    'max_depth':        randint(2, 5),
    'min_child_weight': randint(10, 30),
    'reg_alpha':        uniform(1, 5),
    'reg_lambda':       uniform(3, 10),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.5, 0.5),
}

xgb_reg_search = RandomizedSearchCV(
    XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    xgb_reg_grid,
    n_iter=20, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
xgb_reg_search.fit(X_train, y_train)

# --- Report ---
from sklearn.model_selection import cross_validate

print("\n" + "=" * 50)
for name, search in [('Random Forest', rf_reg_search), ('XGBoost', xgb_reg_search)]:
    cv = cross_validate(search.best_estimator_, X_train, y_train, cv=5,
                        scoring='roc_auc', return_train_score=True)
    print(f"{name}")
    print(f"  Best params:     {search.best_params_}")
    print(f"  ROC-AUC (CV):    {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")
    print(f"  ROC-AUC (Train): {cv['train_score'].mean():.3f}")
    print(f"  Overfit Gap:     {cv['train_score'].mean() - cv['test_score'].mean():.3f}")
    print()

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Random Forest
  Best params:     {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 11, 'min_samples_split': 29}
  ROC-AUC (CV):    0.746 ± 0.033
  ROC-AUC (Train): 0.815
  Overfit Gap:     0.068

XGBoost
  Best params:     {'colsample_bytree': np.float64(0.855670976374325), 'max_depth': 4, 'min_child_weight': 10, 'reg_alpha': np.float64(4.029799873905057), 'reg_lambda': np.float64(12.26300878513349), 'subsample': np.float64(0.8604308102007778)}
  ROC-AUC (CV):    0.756 ± 0.041
  ROC-AUC (Train): 0.827
  Overfit Gap:     0.071



In [18]:
# Full hyperparameter search for Random Forest and XGBoost.
# 100 iterations each, 5-fold CV, optimizing ROC-AUC.
# Builds on regularization pass — search ranges informed by best params found above.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

# --- Random Forest ---
rf_grid = {
    'n_estimators':      randint(100, 500),
    'max_depth':         randint(3, 8),
    'min_samples_leaf':  randint(5, 50),
    'min_samples_split': randint(5, 50),
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    rf_grid,
    n_iter=100, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train)

# --- XGBoost ---
xgb_grid = {
    'n_estimators':      randint(100, 400),
    'max_depth':         randint(2, 5),
    'learning_rate':     uniform(0.01, 0.1),
    'min_child_weight':  randint(10, 30),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.5, 0.5),
    'reg_alpha':         uniform(1, 5),
    'reg_lambda':        uniform(3, 10),
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    xgb_grid,
    n_iter=100, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_train, y_train)

# --- Report ---
print("\n" + "=" * 50)
for name, search in [('Random Forest', rf_search), ('XGBoost', xgb_search)]:
    cv = cross_validate(search.best_estimator_, X_train, y_train, cv=5,
                        scoring='roc_auc', return_train_score=True)
    print(f"{name}")
    print(f"  Best params:     {search.best_params_}")
    print(f"  ROC-AUC (CV):    {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")
    print(f"  ROC-AUC (Train): {cv['train_score'].mean():.3f}")
    print(f"  Overfit Gap:     {cv['train_score'].mean() - cv['test_score'].mean():.3f}")
    print()

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Random Forest
  Best params:     {'max_depth': 4, 'max_features': 0.5, 'min_samples_leaf': 5, 'min_samples_split': 15, 'n_estimators': 250}
  ROC-AUC (CV):    0.749 ± 0.038
  ROC-AUC (Train): 0.855
  Overfit Gap:     0.106

XGBoost
  Best params:     {'colsample_bytree': np.float64(0.7696887242000312), 'learning_rate': np.float64(0.0783963769398141), 'max_depth': 4, 'min_child_weight': 11, 'n_estimators': 259, 'reg_alpha': np.float64(5.701151207124788), 'reg_lambda': np.float64(8.75474177875879), 'subsample': np.float64(0.7552679704826087)}
  ROC-AUC (CV):    0.756 ± 0.033
  ROC-AUC (Train): 0.823
  Overfit Gap:     0.067



In [19]:
# Hyperparameter tuning for Logistic Regression and SVM.
# 100 iterations each, 5-fold CV, optimizing ROC-AUC.
# Both wrapped in StandardScaler pipelines.

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from scipy.stats import uniform, loguniform

# --- Logistic Regression ---
lr_grid = {
    'clf__C':       loguniform(0.001, 100),
    'clf__penalty': ['l1', 'l2'],
    'clf__solver':  ['liblinear', 'saga'],
}

lr_search = RandomizedSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=42))]),
    lr_grid,
    n_iter=100, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
lr_search.fit(X_train, y_train)

# --- SVM ---
svm_grid = {
    'clf__C':     loguniform(0.01, 100),
    'clf__gamma': ['scale', 'auto'],
    'clf__kernel':['rbf', 'linear'],
}

svm_search = RandomizedSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
    svm_grid,
    n_iter=100, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
svm_search.fit(X_train, y_train)

# --- Report ---
print("\n" + "=" * 50)
for name, search in [('Logistic Regression', lr_search), ('SVM', svm_search)]:
    cv = cross_validate(search.best_estimator_, X_train, y_train, cv=5,
                        scoring='roc_auc', return_train_score=True)
    print(f"{name}")
    print(f"  Best params:     {search.best_params_}")
    print(f"  ROC-AUC (CV):    {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")
    print(f"  ROC-AUC (Train): {cv['train_score'].mean():.3f}")
    print(f"  Overfit Gap:     {cv['train_score'].mean() - cv['test_score'].mean():.3f}")
    print()

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Logistic Regression
  Best params:     {'clf__C': np.float64(0.06578820119123178), 'clf__penalty': 'l1', 'clf__solver': 'saga'}
  ROC-AUC (CV):    0.716 ± 0.041
  ROC-AUC (Train): 0.746
  Overfit Gap:     0.030

SVM
  Best params:     {'clf__C': np.float64(0.4673518999562751), 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
  ROC-AUC (CV):    0.721 ± 0.035
  ROC-AUC (Train): 0.827
  Overfit Gap:     0.107



In [20]:
# Final evaluation of all four tuned models on the held-out test set.
# Reports ROC-AUC, log loss, accuracy, precision, recall, F1, and overfit gap.

from sklearn.metrics import (roc_auc_score, log_loss, accuracy_score,
                             precision_score, recall_score, f1_score,
                             classification_report)
from sklearn.model_selection import cross_validate

models_final = {
    'Logistic Regression': lr_search.best_estimator_,
    'SVM':                 svm_search.best_estimator_,
    'Random Forest':       rf_search.best_estimator_,
    'XGBoost':             xgb_search.best_estimator_,
}

summary = []
for name, model in models_final.items():
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    cv = cross_validate(model, X_train, y_train, cv=5,
                        scoring='roc_auc', return_train_score=True)
    print(f"{'='*52}")
    print(f"{name}")
    print(f"{'='*52}")
    print(f"  ROC-AUC (CV):    {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")
    print(f"  ROC-AUC (Train): {cv['train_score'].mean():.3f}")
    print(f"  ROC-AUC (Test):  {roc_auc_score(y_test, y_proba):.3f}")
    print(f"  Overfit Gap:     {cv['train_score'].mean() - cv['test_score'].mean():.3f}")
    print(f"  Log Loss (Test): {log_loss(y_test, y_proba):.3f}")
    print(f"  Accuracy (Test): {accuracy_score(y_test, y_pred):.3f}")
    print(f"  Precision (Test):{precision_score(y_test, y_pred):.3f}")
    print(f"  Recall (Test):   {recall_score(y_test, y_pred):.3f}")
    print(f"  F1 (Test):       {f1_score(y_test, y_pred):.3f}")
    print(f"\n{classification_report(y_test, y_pred)}\n")
    summary.append({
        'Model':           name,
        'ROC-AUC (CV)':    cv['test_score'].mean(),
        'ROC-AUC (Train)': cv['train_score'].mean(),
        'ROC-AUC (Test)':  roc_auc_score(y_test, y_proba),
        'Overfit Gap':     cv['train_score'].mean() - cv['test_score'].mean(),
        'Log Loss':        log_loss(y_test, y_proba),
        'Accuracy':        accuracy_score(y_test, y_pred),
        'Precision':       precision_score(y_test, y_pred),
        'Recall':          recall_score(y_test, y_pred),
        'F1':              f1_score(y_test, y_pred),
    })

print("=" * 52)
print("FINAL COMPARISON (Test Set)")
print("=" * 52)
df_final = pd.DataFrame(summary).set_index('Model').round(3)
print(df_final.to_string())

Logistic Regression
  ROC-AUC (CV):    0.716 ± 0.041
  ROC-AUC (Train): 0.746
  ROC-AUC (Test):  0.769
  Overfit Gap:     0.030
  Log Loss (Test): 0.598
  Accuracy (Test): 0.728
  Precision (Test):0.736
  Recall (Test):   0.662
  F1 (Test):       0.697

              precision    recall  f1-score   support

         0.0       0.72      0.79      0.75        89
         1.0       0.74      0.66      0.70        80

    accuracy                           0.73       169
   macro avg       0.73      0.72      0.73       169
weighted avg       0.73      0.73      0.73       169


SVM
  ROC-AUC (CV):    0.721 ± 0.035
  ROC-AUC (Train): 0.827
  ROC-AUC (Test):  0.794
  Overfit Gap:     0.107
  Log Loss (Test): 0.569
  Accuracy (Test): 0.698
  Precision (Test):0.738
  Recall (Test):   0.562
  F1 (Test):       0.638

              precision    recall  f1-score   support

         0.0       0.68      0.82      0.74        89
         1.0       0.74      0.56      0.64        80

    accuracy    

Now, we try Catboost

In [23]:
# CatBoost lightweight regularization pass — 20 iterations

from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from scipy.stats import randint, uniform

cb_reg_grid = {
    'depth':             randint(2, 6),
    'learning_rate':     uniform(0.01, 0.1),
    'l2_leaf_reg':       uniform(3, 10),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bylevel': uniform(0.5, 0.5),
}

cb_reg_search = RandomizedSearchCV(
    CatBoostClassifier(iterations=100, random_state=42, verbose=0),
    cb_reg_grid,
    n_iter=20, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
cb_reg_search.fit(X_train, y_train)

# Rebuild model with best params as plain Python floats to avoid sklearn clone error
best_params_reg = {k: float(v) if hasattr(v, 'item') else v
                   for k, v in cb_reg_search.best_params_.items()}

cb_reg_best = CatBoostClassifier(
    iterations=100, random_state=42, verbose=0, **best_params_reg
)

cv = cross_validate(cb_reg_best, X_train, y_train, cv=5,
                    scoring='roc_auc', return_train_score=True)

print("\n" + "=" * 50)
print("CatBoost — Regularization Pass")
print(f"  Best params:     {best_params_reg}")
print(f"  ROC-AUC (CV):    {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")
print(f"  ROC-AUC (Train): {cv['train_score'].mean():.3f}")
print(f"  Overfit Gap:     {cv['train_score'].mean() - cv['test_score'].mean():.3f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits

CatBoost — Regularization Pass
  Best params:     {'colsample_bylevel': 0.7989499894055425, 'depth': 5, 'l2_leaf_reg': 6.265407688058354, 'learning_rate': 0.06704439744053994, 'subsample': 0.8083337040103294}
  ROC-AUC (CV):    0.759 ± 0.048
  ROC-AUC (Train): 0.924
  Overfit Gap:     0.165


In [24]:
# CatBoost full hyperparameter search — 100 iterations

from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy.stats import randint, uniform

cb_grid = {
    'depth':             randint(2, 6),
    'learning_rate':     uniform(0.01, 0.1),
    'iterations':        randint(100, 400),
    'l2_leaf_reg':       uniform(3, 10),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bylevel': uniform(0.5, 0.5),
}

cb_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cb_grid,
    n_iter=100, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
cb_search.fit(X_train, y_train)

# Rebuild model with best params as plain Python floats to avoid sklearn clone error
best_params_cb = {k: int(v) if isinstance(v, (int,)) or (hasattr(v, 'item') and 'int' in type(v).__name__)
                  else float(v) if hasattr(v, 'item') else v
                  for k, v in cb_search.best_params_.items()}

cb_best = CatBoostClassifier(random_state=42, verbose=0, **best_params_cb)

cv = cross_validate(cb_best, X_train, y_train, cv=5,
                    scoring='roc_auc', return_train_score=True)

# Test set evaluation
cb_best.fit(X_train, y_train)
y_pred_cb  = cb_best.predict(X_test)
y_proba_cb = cb_best.predict_proba(X_test)[:, 1]

print("=" * 52)
print("CatBoost — Final Evaluation")
print("=" * 52)
print(f"  Best params:     {best_params_cb}")
print(f"  ROC-AUC (CV):    {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")
print(f"  ROC-AUC (Train): {cv['train_score'].mean():.3f}")
print(f"  ROC-AUC (Test):  {roc_auc_score(y_test, y_proba_cb):.3f}")
print(f"  Overfit Gap:     {cv['train_score'].mean() - cv['test_score'].mean():.3f}")
print(f"  Log Loss:        {log_loss(y_test, y_proba_cb):.3f}")
print(f"  Accuracy:        {accuracy_score(y_test, y_pred_cb):.3f}")
print(f"  Precision:       {precision_score(y_test, y_pred_cb):.3f}")
print(f"  Recall:          {recall_score(y_test, y_pred_cb):.3f}")
print(f"  F1:              {f1_score(y_test, y_pred_cb):.3f}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits
CatBoost — Final Evaluation
  Best params:     {'colsample_bylevel': 0.7247253370691017, 'depth': 2, 'iterations': 147, 'l2_leaf_reg': 6.708182521982664, 'learning_rate': 0.07688412526636072, 'subsample': 0.8663689426469987}
  ROC-AUC (CV):    0.761 ± 0.053
  ROC-AUC (Train): 0.867
  ROC-AUC (Test):  0.803
  Overfit Gap:     0.106
  Log Loss:        0.563
  Accuracy:        0.734
  Precision:       0.697
  Recall:          0.775
  F1:              0.734


Full comparison — Temporal Split 2015-2019, after tuning:

In [25]:
# Summary comparison table — Temporal Split 2015-2019
import pandas as pd

summary_all = [
    {'Model': 'Logistic Regression', 'ROC-AUC (CV)': 0.716, 'ROC-AUC (Train)': 0.746, 'ROC-AUC (Test)': 0.769, 'Overfit Gap': 0.030, 'Log Loss': 0.598, 'Accuracy': 0.728, 'Precision': 0.736, 'Recall': 0.662, 'F1': 0.697},
    {'Model': 'SVM',                 'ROC-AUC (CV)': 0.721, 'ROC-AUC (Train)': 0.827, 'ROC-AUC (Test)': 0.794, 'Overfit Gap': 0.107, 'Log Loss': 0.569, 'Accuracy': 0.698, 'Precision': 0.738, 'Recall': 0.562, 'F1': 0.638},
    {'Model': 'Random Forest',       'ROC-AUC (CV)': 0.749, 'ROC-AUC (Train)': 0.855, 'ROC-AUC (Test)': 0.785, 'Overfit Gap': 0.106, 'Log Loss': 0.561, 'Accuracy': 0.710, 'Precision': 0.682, 'Recall': 0.725, 'F1': 0.703},
    {'Model': 'XGBoost',             'ROC-AUC (CV)': 0.756, 'ROC-AUC (Train)': 0.823, 'ROC-AUC (Test)': 0.782, 'Overfit Gap': 0.067, 'Log Loss': 0.563, 'Accuracy': 0.716, 'Precision': 0.690, 'Recall': 0.725, 'F1': 0.707},
    {'Model': 'CatBoost',            'ROC-AUC (CV)': 0.761, 'ROC-AUC (Train)': 0.867, 'ROC-AUC (Test)': 0.803, 'Overfit Gap': 0.106, 'Log Loss': 0.563, 'Accuracy': 0.734, 'Precision': 0.697, 'Recall': 0.775, 'F1': 0.734},
]

df_summary = pd.DataFrame(summary_all).set_index('Model').round(3)

# Styled display in notebook
df_summary.style \
    .background_gradient(subset=['ROC-AUC (CV)', 'ROC-AUC (Test)'], cmap='Greens') \
    .background_gradient(subset=['Overfit Gap'], cmap='Reds') \
    .format('{:.3f}') \
    .set_caption('Model Comparison — Temporal Split 2015-2019')

,ROC-AUC (CV),ROC-AUC (Train),ROC-AUC (Test),Overfit Gap,Log Loss,Accuracy,Precision,Recall,F1
Model,,,,,,,,,
Logistic Regression,0.716,0.746,0.769,0.030,0.598,0.728,0.736,0.662,0.697
SVM,0.721,0.827,0.794,0.107,0.569,0.698,0.738,0.562,0.638
Random Forest,0.749,0.855,0.785,0.106,0.561,0.710,0.682,0.725,0.703
XGBoost,0.756,0.823,0.782,0.067,0.563,0.716,0.690,0.725,0.707
CatBoost,0.761,0.867,0.803,0.106,0.563,0.734,0.697,0.775,0.734
